# Hardware Signal Generator

Generates every input sequence needed to run notebooks 01–04 against a physical DUT, saves them to `data/hardware/inputs/`, and gives step-by-step recording instructions for each one. Run this notebook once before beginning hardware testing.

Input files are deterministic given the parameters below (they mirror the exact generator calls used in notebooks 01–04) — re-running this notebook regenerates identical files.

In [1]:
# ── Hardware Signal Generator ─────────────────────────────────────────────────
import csv
import json
import numpy as np
from pathlib import Path

from prc_toolkit.config import DT, FS, SEED
from prc_toolkit.signals.generators import (
    multisine, sine_sweep, dc_near_zero, iid_uniform,
    poisson_spike_train, delayed_spike_train, bias_positive
)

# Load Section 1 results for V_SAFE
with open("results/section1_results.json") as f:
    s1 = json.load(f)
V_SAFE = s1["V_SAFE"]

# Output directories
INPUT_DIR  = Path("data/hardware/inputs")
OUTPUT_DIR = Path("data/hardware/outputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"V_SAFE = {V_SAFE:.4f}")
print(f"Sampling rate: {FS} Hz  (DT = {DT}s)")

V_SAFE = 0.8000
Sampling rate: 100 Hz  (DT = 0.01s)


In [2]:
# ── Helper: save an input sequence as a CSV the wrapped notebooks expect ──────
def save_input(filename: str, u_seq: np.ndarray):
    """
    Save an input sequence to data/hardware/inputs/.
    u_seq: shape (T,) or (T, N_u). Saved as CSV with header u0, u1, ...
    Uses the stdlib csv module -- this toolkit has no pandas dependency.
    """
    u = np.asarray(u_seq)
    if u.ndim == 1:
        u = u.reshape(-1, 1)
    header = [f"u{i}" for i in range(u.shape[1])]
    with open(INPUT_DIR / filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(u.tolist())
    print(f"Saved: {INPUT_DIR / filename}  ({u.shape[0]} timesteps)")

## Notebook 01 — Fundamentals

Sections 1.1 and 1.2 are demonstration-only and do not support hardware mode
(see the note at the top of notebook 01). Section 1.3 (Distortion and Noise
Analysis) does support `DUT_MODEL = "hardware"`.

The input is a multisine at frequencies [1, 3, 7, 11] Hz, amplitude V_SAFE,
8 one-second periods (matches notebook 01's N_PERIODS=8, PERIOD_DURATION=1.0s).
Sampling rate: 100 Hz.

**To use:** reset the DUT, then loop `section1_multisine.csv` back-to-back
until the output has settled (per the toolkit's standard settling criterion).
Once settled, immediately play it one more full pass and record **only that
pass** as `data/hardware/outputs/section1_multisine.csv`. Section 1.3.2
(Noise Floor Analysis) reuses this same recording -- no separate file needed.

In [3]:
# Section 1 multisine (reference only -- see note above)
N_PERIODS_S1 = 8
PERIOD_DURATION_S1 = 1.0
u_s1_multisine = multisine(duration=N_PERIODS_S1 * PERIOD_DURATION_S1, amplitude=V_SAFE, fs=FS)
save_input("section1_multisine.csv", u_s1_multisine)

Saved: data/hardware/inputs/section1_multisine.csv  (800 timesteps)


## Notebook 02 — System Identification

Section 2 requires several separate recordings. For each one, follow the
"settle" / "probe" split below where applicable:

1. **Fingerprint sweep (2.1):** one recording per amplitude step. Reset the
   DUT, play the input file continuously, but discard the settle-period
   output described below -- record only the remainder.
2. **FMP / ESP (2.3):** each trial has a *settle* input (played and discarded,
   not recorded) followed by a shared *probe* input (played and recorded).
   Reset the DUT before each trial's settle input.
3. **SP test (2.4):** loop the washout input until the DUT's output has
   settled (not recorded), then immediately play the trial input and record
   the entire output.

Name your recorded output files exactly as instructed in each subsection
below and save them to `data/hardware/outputs/`.

In [4]:
# ── 2.1 Fingerprint sweep inputs ───────────────────────────────────────────
N_AMP_STEPS_FP    = 10
FP_SETTLE_PERIODS = 2   # seconds (1 Hz sine) driven but discarded, let the transient settle
FP_RECORD_PERIODS = 3   # seconds (1 Hz sine) recorded after settling

fp_duration = float(FP_SETTLE_PERIODS + FP_RECORD_PERIODS)
n_settle_fp = int(FP_SETTLE_PERIODS * FS)
sweep = sine_sweep(duration=fp_duration, amplitude=V_SAFE, n_steps=N_AMP_STEPS_FP, fs=FS)
for amp_idx, (amp_linear, u_step) in enumerate(sweep):
    save_input(f"section2_fingerprint_amp{amp_idx:02d}.csv", u_step)
print(f"\nFor each file above: reset the DUT and play the file continuously. "
      f"Discard the first {n_settle_fp} samples of output ({FP_SETTLE_PERIODS:.0f}s "
      f"settling). Record the remaining {int(FP_RECORD_PERIODS*FS)} samples as "
      f"data/hardware/outputs/section2_fingerprint_amp{{i:02d}}.csv (same index i).")

Saved: data/hardware/inputs/section2_fingerprint_amp00.csv  (500 timesteps)
Saved: data/hardware/inputs/section2_fingerprint_amp01.csv  (500 timesteps)
Saved: data/hardware/inputs/section2_fingerprint_amp02.csv  (500 timesteps)
Saved: data/hardware/inputs/section2_fingerprint_amp03.csv  (500 timesteps)
Saved: data/hardware/inputs/section2_fingerprint_amp04.csv  (500 timesteps)
Saved: data/hardware/inputs/section2_fingerprint_amp05.csv  (500 timesteps)
Saved: data/hardware/inputs/section2_fingerprint_amp06.csv  (500 timesteps)
Saved: data/hardware/inputs/section2_fingerprint_amp07.csv  (500 timesteps)
Saved: data/hardware/inputs/section2_fingerprint_amp08.csv  (500 timesteps)
Saved: data/hardware/inputs/section2_fingerprint_amp09.csv  (500 timesteps)

For each file above: reset the DUT and play the file continuously. Discard the first 200 samples of output (2s settling). Record the remaining 300 samples as data/hardware/outputs/section2_fingerprint_amp{i:02d}.csv (same index i).


In [5]:
# ── 2.3 FMP settle + probe inputs ──────────────────────────────────────────
N_TRIALS = 2  # from prc_toolkit.config.N_TRIALS
SETTLE_DURATION = 10.0
PROBE_DURATION  = 10.0

u_fmp_probe = dc_near_zero(duration=PROBE_DURATION, fs=FS)
save_input("section2_fmp_probe.csv", u_fmp_probe)

for trial_i in range(N_TRIALS):
    rng_trial = np.random.default_rng(trial_i + 100)
    phases = rng_trial.uniform(0, 2 * np.pi, 4)
    t = np.arange(int(SETTLE_DURATION * FS)) * DT
    u_A = V_SAFE * 0.5 * (
        np.sin(2*np.pi*1*t  + phases[0]) +
        np.sin(2*np.pi*3*t  + phases[1]) +
        np.sin(2*np.pi*7*t  + phases[2]) +
        np.sin(2*np.pi*11*t + phases[3])
    )
    save_input(f"section2_fmp_settle_{trial_i:02d}.csv", u_A)

print("\nFor each trial i: reset the DUT, play section2_fmp_settle_{i:02d}.csv "
      "WITHOUT recording, then immediately play section2_fmp_probe.csv and "
      "record its output as data/hardware/outputs/section2_fmp_trial_{i:02d}.csv.")

Saved: data/hardware/inputs/section2_fmp_probe.csv  (1000 timesteps)
Saved: data/hardware/inputs/section2_fmp_settle_00.csv  (1000 timesteps)
Saved: data/hardware/inputs/section2_fmp_settle_01.csv  (1000 timesteps)

For each trial i: reset the DUT, play section2_fmp_settle_{i:02d}.csv WITHOUT recording, then immediately play section2_fmp_probe.csv and record its output as data/hardware/outputs/section2_fmp_trial_{i:02d}.csv.


In [6]:
# ── 2.3 ESP settle + probe inputs ──────────────────────────────────────────
ESP_SWAP_DURATION = 10.0

t_C = np.arange(int(ESP_SWAP_DURATION * FS)) * DT
u_esp_probe = V_SAFE * 0.4 * (
    np.sin(2*np.pi*2*t_C) +
    np.sin(2*np.pi*5*t_C) +
    np.sin(2*np.pi*9*t_C)
)
save_input("section2_esp_probe.csv", u_esp_probe)

for trial_i in range(N_TRIALS):
    rng_trial = np.random.default_rng(trial_i + 200)
    phases = rng_trial.uniform(0, 2 * np.pi, 4)
    t = np.arange(int(SETTLE_DURATION * FS)) * DT
    u_A = V_SAFE * 0.5 * (
        np.sin(2*np.pi*1*t  + phases[0]) +
        np.sin(2*np.pi*3*t  + phases[1]) +
        np.sin(2*np.pi*7*t  + phases[2]) +
        np.sin(2*np.pi*11*t + phases[3])
    )
    save_input(f"section2_esp_settle_{trial_i:02d}.csv", u_A)

print("\nFor each trial i: reset the DUT, play section2_esp_settle_{i:02d}.csv "
      "WITHOUT recording, then immediately play section2_esp_probe.csv and "
      "record its output as data/hardware/outputs/section2_esp_trial_{i:02d}.csv.")

Saved: data/hardware/inputs/section2_esp_probe.csv  (1000 timesteps)
Saved: data/hardware/inputs/section2_esp_settle_00.csv  (1000 timesteps)
Saved: data/hardware/inputs/section2_esp_settle_01.csv  (1000 timesteps)

For each trial i: reset the DUT, play section2_esp_settle_{i:02d}.csv WITHOUT recording, then immediately play section2_esp_probe.csv and record its output as data/hardware/outputs/section2_esp_trial_{i:02d}.csv.


In [7]:
# ── 2.4 SP test: washout + trial inputs ────────────────────────────────────
WASHOUT_FREQ = 5.0
WASHOUT_AMP  = V_SAFE
t_washout = np.arange(0, 1.0, DT)
u_washout = WASHOUT_AMP * np.sin(2 * np.pi * WASHOUT_FREQ * t_washout)
save_input("section2_washout.csv", u_washout)

POISSON_RATE_HZ = 5.0
SPIKE_DURATION  = 15.0
PULSE_WIDTH     = 2
SPIKE_IDX       = int(2.0 * FS)
DELAY_SAMPLES   = int(0.1 * FS)
SEED_SPIKES     = 7

u_template = poisson_spike_train(
    duration=SPIKE_DURATION, rate_hz=POISSON_RATE_HZ, amplitude=V_SAFE,
    pulse_width_samples=PULSE_WIDTH, fs=FS, seed=SEED_SPIKES
)
u_variant = delayed_spike_train(u_template=u_template, spike_idx=SPIKE_IDX, delay_samples=DELAY_SAMPLES)
save_input("section2_sp_trial_00_input.csv", u_template)
save_input("section2_sp_trial_01_input.csv", u_variant)

print("\nFor each trial: reset the DUT, loop section2_washout.csv until the "
      "output has settled (not recorded), then immediately play "
      "section2_sp_trial_{i:02d}_input.csv and record the ENTIRE output as "
      "data/hardware/outputs/section2_sp_trial_{i:02d}.csv (i=00 for the "
      "template, i=01 for the variant).")

Saved: data/hardware/inputs/section2_washout.csv  (100 timesteps)
Saved: data/hardware/inputs/section2_sp_trial_00_input.csv  (1500 timesteps)
Saved: data/hardware/inputs/section2_sp_trial_01_input.csv  (1500 timesteps)

For each trial: reset the DUT, loop section2_washout.csv until the output has settled (not recorded), then immediately play section2_sp_trial_{i:02d}_input.csv and record the ENTIRE output as data/hardware/outputs/section2_sp_trial_{i:02d}.csv (i=00 for the template, i=01 for the variant).


## Notebook 03 — System Characterization

Section 3 uses a single pair of recordings, shared across Sections 3.2–3.6:
one trial from initial condition A, one from initial condition B. Each trial
has a settle input (played and discarded) followed by the same shared drive
input (played and recorded).

In [8]:
# ── Section 3 shared-run inputs ────────────────────────────────────────────
SIGNAL_DURATION_S3 = 30.0
SETTLE_DURATION_S3 = 5.0
SEED_UNIFORM_0 = 99

u_drive = iid_uniform(duration=SIGNAL_DURATION_S3, amplitude=V_SAFE, fs=FS, seed=SEED_UNIFORM_0)
save_input("section3_drive.csv", u_drive)

u_settle_A = iid_uniform(duration=SETTLE_DURATION_S3, amplitude=V_SAFE*0.3, fs=FS, seed=1001)
save_input("section3_settle_trial00.csv", u_settle_A)

u_settle_B = iid_uniform(duration=SETTLE_DURATION_S3, amplitude=V_SAFE*0.3, fs=FS, seed=1002)
save_input("section3_settle_trial01.csv", u_settle_B)

print("\nTrial 0: reset the DUT, play section3_settle_trial00.csv WITHOUT "
      "recording, then immediately play section3_drive.csv and record its "
      "output as data/hardware/outputs/section3_shared_run_trial00.csv.\n"
      "Trial 1: repeat with section3_settle_trial01.csv, recording as "
      "section3_shared_run_trial01.csv.")

Saved: data/hardware/inputs/section3_drive.csv  (3000 timesteps)
Saved: data/hardware/inputs/section3_settle_trial00.csv  (500 timesteps)
Saved: data/hardware/inputs/section3_settle_trial01.csv  (500 timesteps)

Trial 0: reset the DUT, play section3_settle_trial00.csv WITHOUT recording, then immediately play section3_drive.csv and record its output as data/hardware/outputs/section3_shared_run_trial00.csv.
Trial 1: repeat with section3_settle_trial01.csv, recording as section3_shared_run_trial01.csv.


## Notebook 04 — Benchmarks

Each benchmark needs one continuous recording: a washout prefix (played but
not recorded) followed by the task's driving signal (recorded). Training
happens offline in notebook 04 after recording — you do not need to retrain
between recordings, only between benchmarks.

For each benchmark below, play the saved input file continuously from the
start. Start recording only after the stated number of washout samples have
been played, and save the recorded remainder under the stated filename.

In [9]:
# ── 4.1 NARMA-10 input ──────────────────────────────────────────────────────
N_WASHOUT = 200
NARMA_ORDER = 10
NARMA_ALPHA, NARMA_BETA, NARMA_GAMMA, NARMA_DELTA = 0.3, 0.05, 1.5, 0.1
N_TRAIN, N_TEST = 5000, 1000

rng = np.random.default_rng(SEED)
T_total = N_WASHOUT + N_TRAIN + N_TEST + NARMA_ORDER
u_narma = rng.uniform(0, 0.5, T_total)
save_input("section4_narma_input.csv", u_narma)

skip = N_WASHOUT + NARMA_ORDER
print(f"\nPlay section4_narma_input.csv from the start. Discard the first "
      f"{skip} samples of output ({skip*DT:.1f}s of washout). Record the "
      f"remaining {T_total - 1 - skip} samples as "
      f"data/hardware/outputs/section4_narma_run.csv.")

Saved: data/hardware/inputs/section4_narma_input.csv  (6210 timesteps)

Play section4_narma_input.csv from the start. Discard the first 210 samples of output (2.1s of washout). Record the remaining 5999 samples as data/hardware/outputs/section4_narma_run.csv.


In [10]:
# ── 4.2 Mackey-Glass input ──────────────────────────────────────────────────
MG_BETA, MG_GAMMA, MG_N, MG_TAU = 0.2, 0.1, 10, 17
MG_SUBSAMPLE, MG_X0, MG_H = 10, 0.5, 1
dt_mg = 0.1
tau_steps = int(MG_TAU / dt_mg)
T_fine = (N_WASHOUT + N_TRAIN + N_TEST + MG_H) * MG_SUBSAMPLE + tau_steps

mg = np.full(T_fine, MG_X0)
for t in range(tau_steps, T_fine - 1):
    mg[t+1] = mg[t] + dt_mg * (
        MG_BETA * mg[t - tau_steps] / (1 + mg[t - tau_steps] ** MG_N)
        - MG_GAMMA * mg[t]
    )
mg_series = mg[tau_steps::MG_SUBSAMPLE]
u_mg = mg_series[:-MG_H]
save_input("section4_mackeyglass_input.csv", u_mg)

print(f"\nPlay section4_mackeyglass_input.csv from the start. Discard the "
      f"first {N_WASHOUT} samples of output. Record the remaining "
      f"{len(u_mg) - N_WASHOUT} samples as "
      f"data/hardware/outputs/section4_mackeyglass_run.csv.")

Saved: data/hardware/inputs/section4_mackeyglass_input.csv  (6200 timesteps)

Play section4_mackeyglass_input.csv from the start. Discard the first 200 samples of output. Record the remaining 6000 samples as data/hardware/outputs/section4_mackeyglass_run.csv.


In [11]:
# ── 4.3 Lorenz'\''63 input ──────────────────────────────────────────────────
LZ_SIGMA, LZ_RHO, LZ_BETA, LZ_DT = 10.0, 28.0, 8.0/3.0, 0.01
LZ_X0 = (1.0, 1.0, 1.0)
T_lz = N_WASHOUT + N_TRAIN + N_TEST + 1
xyz = np.zeros((T_lz, 3))
xyz[0] = LZ_X0
for t in range(T_lz - 1):
    x, y, z = xyz[t]
    xyz[t+1, 0] = x + LZ_DT * LZ_SIGMA * (y - x)
    xyz[t+1, 1] = y + LZ_DT * (x * (LZ_RHO - z) - y)
    xyz[t+1, 2] = z + LZ_DT * (x * y - LZ_BETA * z)

u_lz = xyz[:-1, 0]
lz_amplitude = float(np.max(np.abs(u_lz)))
u_seq_lz = bias_positive(u_lz, lz_amplitude)  # if your DUT needs non-negative input; omit otherwise
save_input("section4_lorenz_input.csv", u_lz)

print(f"\nPlay section4_lorenz_input.csv from the start. Discard the first "
      f"{N_WASHOUT} samples of output. Record the remaining "
      f"{len(u_lz) - N_WASHOUT} samples as "
      f"data/hardware/outputs/section4_lorenz_run.csv.\n"
      f"Note: this input is bipolar (range ~+/-{lz_amplitude:.1f}). If your DUT "
      f"requires non-negative input, apply the same bias_positive() "
      f"transform notebook 04 uses before playing it.")

Saved: data/hardware/inputs/section4_lorenz_input.csv  (6200 timesteps)

Play section4_lorenz_input.csv from the start. Discard the first 200 samples of output. Record the remaining 6000 samples as data/hardware/outputs/section4_lorenz_run.csv.
Note: this input is bipolar (range ~+/-21.2). If your DUT requires non-negative input, apply the same bias_positive() transform notebook 04 uses before playing it.


In [12]:
# ── 4.4 Sunspot input ───────────────────────────────────────────────────────
import csv as _csv
SUNSPOT_PATH = Path("data/sunspot_monthly.csv")
ss_values = []
with open(SUNSPOT_PATH, newline="") as f:
    reader = _csv.DictReader(f)
    for row in reader:
        sn = float(row["sunspot_number"])
        if sn == -1:
            continue
        ss_values.append(sn)
ss_values = np.array(ss_values)
ss_norm = (ss_values - ss_values.min()) / (ss_values.max() - ss_values.min())
u_ss = ss_norm[:-1]
save_input("section4_sunspot_input.csv", u_ss)

SUNSPOT_WASHOUT = 50
print(f"\nPlay section4_sunspot_input.csv from the start. Discard the first "
      f"{SUNSPOT_WASHOUT} samples of output. Record the remaining "
      f"{len(u_ss) - SUNSPOT_WASHOUT} samples as "
      f"data/hardware/outputs/section4_sunspot_run.csv.")

Saved: data/hardware/inputs/section4_sunspot_input.csv  (3330 timesteps)

Play section4_sunspot_input.csv from the start. Discard the first 50 samples of output. Record the remaining 3280 samples as data/hardware/outputs/section4_sunspot_run.csv.


In [13]:
# ── 4.5 XOR input ───────────────────────────────────────────────────────────
XOR_LAG = 2
rng_xor = np.random.default_rng(SEED + 1)
T_xor = N_WASHOUT + N_TRAIN + N_TEST + XOR_LAG
u_xor_raw = rng_xor.integers(0, 2, T_xor).astype(float)
save_input("section4_xor_input.csv", u_xor_raw)

skip_xor = N_WASHOUT + XOR_LAG
print(f"\nPlay section4_xor_input.csv from the start. Discard the first "
      f"{skip_xor} samples of output. Record the remaining "
      f"{T_xor - skip_xor} samples as "
      f"data/hardware/outputs/section4_xor_run.csv.")

Saved: data/hardware/inputs/section4_xor_input.csv  (6202 timesteps)

Play section4_xor_input.csv from the start. Discard the first 202 samples of output. Record the remaining 6000 samples as data/hardware/outputs/section4_xor_run.csv.


## Output format

All recorded output files must be saved to `data/hardware/outputs/` with the
filenames specified above. Each file must be a CSV with:
- A header row with column names `h0`, `h1`, `h2`, ... (one per output electrode)
- One row per timestep, at 100 Hz
- Floating-point voltage values in volts

Your DAQ software will need to export in this format, or you will need to
convert its native output. The toolkit makes no assumptions about your DAQ
hardware — any device capable of 100 Hz synchronized input playback and
output recording is suitable.

Once all output files are saved, run notebooks 02–04 with `DUT_MODEL =
"hardware"` in each config cell, and set `HARDWARE_DATA_PATH =
"data/hardware/outputs"` (instead of the default demo path) to use your real
recordings.